# Generate Multispecies Slow-Mode Inputs

This notebook is the consolidated generation pipeline. It does the expensive stages only and saves reusable outputs:

1. load all-pair distance traces across species,
2. fit/reuse PCA before wavelets,
3. run wavelet transform and PCA after wavelets,
4. optionally assign microstates and compute slow modes from the saved projections.

Plotting and species-identity analyses live in separate notebooks so this file stays focused on generation.

## 1. Imports

In [1]:
from pathlib import Path
import gc
import importlib
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import InconsistentVersionWarning

REPO_ROOT = Path('/Users/meganbishop/slowmodeevo').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import all_pair_distance_data as apd
import pooled_user_pipeline as pup
pup = importlib.reload(pup)


## 2. Configuration

In [2]:
REPO_ROOT = Path('/Users/meganbishop/slowmodeevo').resolve()
DATAVERSE_ROOT = Path('/Users/meganbishop/moseq_dataverse')

RUN_NAME = 'pca_multispecies_evensamp'
OUTPUT_ROOT = REPO_ROOT / 'outputs/single_species_distance_comparison/'
RUN_ROOT = OUTPUT_ROOT / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPRESENTATION_NAME = 'egocentered_and_normalized_distances'
COORDINATE_DATASET = 'recording/egocentric_coordinates_rigid'
DATASET_NAME_AFTER_PRE_PCA = f'{REPRESENTATION_NAME}_pre_wavelet_pca'

SEED = 14
FS_HZ = 120.0
FMIN = 0.5
FMAX_CAP = 60.0
FMAX = min(FMAX_CAP, 0.45 * FS_HZ)
N_FREQS = 25

DISTANCE_PCA_VARIANCE_TARGET = 0.99
DISTANCE_PCA_MAX_COMPONENTS = 100
DISTANCE_PCA_SUBSAMPLE_FACTOR = 50
BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES = True
DISTANCE_PCA_ROWS_PER_SPECIES = None  # None uses the smallest species' available sample count.



POST_WAVELET_PCA_VARIANCE_TARGET = 0.99
POST_WAVELET_PCA_MAX_COMPONENTS = 100
WAVELET_SUBSAMPLE_FACTOR = 50
NORMALIZE_WAVELETS_FOR_PCA = True
BALANCE_POST_WAVELET_PCA_SAMPLES_BY_SPECIES = True
POST_WAVELET_PCA_ROWS_PER_SPECIES = None  # None uses the smallest species' available wavelet sample count.
POST_WAVELET_PCA_FIT_METHOD = 'streaming_covariance'
SAVE_WAVELET_SAMPLES = True 
REUSE_EXISTING = True

RUN_SLOW_MODES = True
D_EMBED = 10
N_CLUSTERS = 1000
KMEANS_SUBSAMPLE_FACTOR = 20
KMEANS_N_INIT = 20
TAU_SECONDS = 2.0
N_BASINS = 4
USE_RECURSIVE_ARMS = False

GLOBAL_ID_SEPARATOR = '__subject__'
print(f'RUN_ROOT = {RUN_ROOT}')


RUN_ROOT = /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp


## 3. Load Distance Data And Build Global IDs

In [3]:
distance_data = apd.AllPairDistanceData(DATAVERSE_ROOT)
base_loader = distance_data.make_loader(COORDINATE_DATASET, REPRESENTATION_NAME)

species_names = distance_data.available_species
rows = []
for species in species_names:
    for subject in distance_data.species_subjects(species):
        rows.append({
            'species': species,
            'individual_id': str(subject),
            'global_id': f'{species}{GLOBAL_ID_SEPARATOR}{subject}',
        })
individual_manifest = pd.DataFrame(rows)
individual_manifest.to_csv(RUN_ROOT / 'multispecies_individual_manifest.csv', index=False)

global_ids = individual_manifest['global_id'].tolist()
print(f'{len(species_names)} species, {len(global_ids)} individuals')
display(individual_manifest.groupby('species').size().rename('n_individuals').reset_index())

8 species, 354 individuals


,species,n_individuals
0,Mus_caroli,15
1,Mus_musculus,93
2,Mus_spretus,16
3,Peromyscus_californicus,23
4,Peromyscus_gossypinus,20
5,Peromyscus_leucopus,14
6,Peromyscus_maniculatus,129
7,Peromyscus_polionotus,44


## 4. PCA Before Wavelet Transform

In [4]:
def parse_global_id(global_id):
    species, individual_id = str(global_id).split(GLOBAL_ID_SEPARATOR, 1)
    return species, individual_id


def distance_pca_config():
    return {
        'run_name': RUN_NAME,
        'representation_name': REPRESENTATION_NAME,
        'coordinate_dataset': COORDINATE_DATASET,
        'global_ids': list(global_ids),
        'distance_pca_variance_target': float(DISTANCE_PCA_VARIANCE_TARGET),
        'distance_pca_max_components': int(DISTANCE_PCA_MAX_COMPONENTS),
        'distance_pca_subsample_factor': int(DISTANCE_PCA_SUBSAMPLE_FACTOR),
        'balance_distance_pca_samples_by_species': bool(BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES),
        'distance_pca_rows_per_species': (
            None if DISTANCE_PCA_ROWS_PER_SPECIES is None else int(DISTANCE_PCA_ROWS_PER_SPECIES)
        ),
        'distance_pca_fit_method': DISTANCE_PCA_FIT_METHOD,
        'seed': int(SEED),
    }


def load_pickle(path):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
        with open(path, 'rb') as handle:
            return pickle.load(handle)


def _available_fixed_stride_rows(n_frames, factor):
    return len(np.arange(0, int(n_frames), max(1, int(factor)), dtype=int))


def _build_species_balanced_distance_pca_plan(sample_rows, rows_per_species=None):
    plan = pd.DataFrame(sample_rows).copy()
    species_available = plan.groupby('species')['available_sample_rows'].sum().sort_index()
    if species_available.empty:
        raise ValueError('No species available for distance PCA sampling')
    if (species_available <= 0).any():
        empty = species_available[species_available <= 0].index.tolist()
        raise ValueError(f'Species with no distance PCA sample rows: {empty}')
    target_rows = int(species_available.min() if rows_per_species is None else rows_per_species)
    target_rows = max(1, target_rows)
    plan['species_sample_budget'] = plan['species'].map(
        lambda species: min(target_rows, int(species_available.loc[species]))
    ).astype(int)
    plan['n_sample_rows'] = 0
    for species, sub in plan.groupby('species', sort=True):
        species_total = int(sub['available_sample_rows'].sum())
        species_budget = min(target_rows, species_total)
        exact = sub['available_sample_rows'].to_numpy(float) * species_budget / species_total
        take = np.floor(exact).astype(int)
        shortfall = int(species_budget - take.sum())
        if shortfall > 0:
            order = np.argsort(-(exact - take), kind='mergesort')[:shortfall]
            take[order] += 1
        take = np.minimum(take, sub['available_sample_rows'].to_numpy(int))
        shortfall = int(species_budget - take.sum())
        if shortfall > 0:
            spare = sub['available_sample_rows'].to_numpy(int) - take
            for local_idx in np.argsort(-spare, kind='mergesort'):
                if shortfall <= 0 or spare[local_idx] <= 0:
                    break
                add = min(shortfall, int(spare[local_idx]))
                take[local_idx] += add
                shortfall -= add
        plan.loc[sub.index, 'n_sample_rows'] = take
    return plan


def _select_distance_pca_rows(X, n_take, factor, rng):
    starts = np.arange(0, int(X.shape[0]), max(1, int(factor)), dtype=int)
    if n_take is None or int(n_take) >= starts.size:
        selected = starts
    else:
        selected = np.sort(rng.choice(starts, size=int(n_take), replace=False))
    return np.asarray(X[selected], dtype=np.float32)


def _iter_distance_pca_batches(sample_plan, *, seed):
    rng = np.random.default_rng(seed)
    for index, row in enumerate(sample_plan.itertuples(index=False), start=1):
        species = str(row.species)
        individual_id = str(row.individual_id)
        global_id = str(row.global_id)
        n_take = int(row.n_sample_rows)
        if n_take <= 0:
            continue
        bundle = base_loader(species, individual_id=individual_id, dataset_name=REPRESENTATION_NAME)
        X_sample = _select_distance_pca_rows(
            bundle['X'],
            n_take,
            DISTANCE_PCA_SUBSAMPLE_FACTOR,
            rng,
        )
        print(f'[{index}/{len(sample_plan)}] covariance sample {global_id}: {X_sample.shape}')
        yield X_sample
        del bundle, X_sample
        gc.collect()


def fit_or_load_distance_pca():
    checkpoint_path = RUN_ROOT / 'pre_wavelet_pca95_checkpoint.pkl'
    sample_manifest_path = RUN_ROOT / 'pre_wavelet_pca_sample_manifest.csv'
    balanced_plan_path = RUN_ROOT / 'pre_wavelet_pca_balanced_sample_plan.csv'
    config = distance_pca_config()
    if REUSE_EXISTING and checkpoint_path.exists():
        cached = load_pickle(checkpoint_path)
        cached_pca = cached.get('pca')
        if cached.get('config') == config and getattr(cached_pca, 'fit_method', None) == 'streaming_covariance':
            print(f'Reusing pre-wavelet covariance PCA: {checkpoint_path}')
            return cached
        print('Found pre-wavelet PCA checkpoint, but config/method changed; refitting.')

    availability_rows = []
    n_features = None
    for index, global_id in enumerate(global_ids, start=1):
        species, individual_id = parse_global_id(global_id)
        bundle = base_loader(species, individual_id=individual_id, dataset_name=REPRESENTATION_NAME)
        if n_features is None:
            n_features = int(bundle['X'].shape[1])
        elif bundle['X'].shape[1] != n_features:
            raise ValueError(
                f'{global_id}: expected {n_features} features, got {bundle["X"].shape[1]}'
            )
        availability_rows.append({
            'global_id': global_id,
            'species': species,
            'individual_id': individual_id,
            'n_raw_frames': int(bundle['X'].shape[0]),
            'available_sample_rows': _available_fixed_stride_rows(
                bundle['X'].shape[0], DISTANCE_PCA_SUBSAMPLE_FACTOR
            ),
            'n_features': int(bundle['X'].shape[1]),
        })
        print(f'[{index}/{len(global_ids)}] distance PCA availability {global_id}: {availability_rows[-1]["available_sample_rows"]:,} rows')
        del bundle
        gc.collect()

    if BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES:
        sample_plan = _build_species_balanced_distance_pca_plan(
            availability_rows,
            rows_per_species=DISTANCE_PCA_ROWS_PER_SPECIES,
        )
    else:
        sample_plan = pd.DataFrame(availability_rows).copy()
        sample_plan['species_sample_budget'] = sample_plan.groupby('species')['available_sample_rows'].transform('sum')
        sample_plan['n_sample_rows'] = sample_plan['available_sample_rows']

    sample_plan.to_csv(balanced_plan_path, index=False)
    species_plan = sample_plan.groupby('species')[['available_sample_rows', 'n_sample_rows']].sum().reset_index()
    print('pre-wavelet covariance PCA sample rows by species:')
    display(species_plan)

    n_fit_components = min(
        int(DISTANCE_PCA_MAX_COMPONENTS),
        int(n_features),
        int(sample_plan['n_sample_rows'].sum()),
    )
    print(f'pre-wavelet covariance PCA: {n_fit_components} components from {n_features} features')
    pca = pup.fit_streaming_covariance_pca(
        _iter_distance_pca_batches(sample_plan, seed=SEED),
        n_components=n_fit_components,
        verbose=True,
        label='pre-wavelet PCA',
    )

    cumulative = np.cumsum(np.asarray(pca.explained_variance_ratio_, dtype=float))
    idx = int(np.searchsorted(cumulative, DISTANCE_PCA_VARIANCE_TARGET, side='left'))
    reached = idx < len(cumulative)
    n_kept = idx + 1 if reached else len(cumulative)
    result = {
        'config': config,
        'pca': pca,
        'n_components': int(n_kept),
        'fit_components': int(pca.n_components_),
        'explained_variance': pca.explained_variance_,
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'cumulative_variance': cumulative,
        'variance_target': float(DISTANCE_PCA_VARIANCE_TARGET),
        'variance_target_reached': bool(reached),
        'sample_shape': (int(sample_plan['n_sample_rows'].sum()), int(n_features)),
        'pca_fit_method': getattr(pca, 'fit_method', DISTANCE_PCA_FIT_METHOD),
        'pca_fit_rows': int(sample_plan['n_sample_rows'].sum()),
        'sample_rows': sample_plan,
        'balance_samples_by_species': bool(BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES),
        'rows_per_species': (
            None if DISTANCE_PCA_ROWS_PER_SPECIES is None else int(DISTANCE_PCA_ROWS_PER_SPECIES)
        ),
        'output_file': str(checkpoint_path),
    }
    with open(checkpoint_path, 'wb') as handle:
        pickle.dump(result, handle)
    sample_plan.to_csv(sample_manifest_path, index=False)
    print(f'Saved pre-wavelet covariance PCA: {checkpoint_path}')
    print(f'Covariance PCA saw {result["pca_fit_rows"]:,} sampled rows total')
    gc.collect()
    return result


distance_pca_result = fit_or_load_distance_pca()
pre_wavelet_pca = distance_pca_result['pca']
pre_wavelet_n_components = int(distance_pca_result['n_components'])
print(f'keeping {pre_wavelet_n_components} pre-wavelet PCs')


[1/354] distance PCA availability Mus_caroli__subject__CAROLI-F-2L-741: 8,641 rows
[2/354] distance PCA availability Mus_caroli__subject__CAROLI-F-L-737: 12,961 rows
[3/354] distance PCA availability Mus_caroli__subject__CAROLI-F-L-741: 12,961 rows
[4/354] distance PCA availability Mus_caroli__subject__CAROLI-F-LR-741: 12,961 rows
[5/354] distance PCA availability Mus_caroli__subject__CAROLI-F-N-740: 12,961 rows
[6/354] distance PCA availability Mus_caroli__subject__CAROLI-F-N-741: 12,961 rows
[7/354] distance PCA availability Mus_caroli__subject__CAROLI-F-R-737: 12,961 rows
[8/354] distance PCA availability Mus_caroli__subject__CAROLI-F-R-741: 12,961 rows
[9/354] distance PCA availability Mus_caroli__subject__CAROLI-M-L-734: 12,961 rows
[10/354] distance PCA availability Mus_caroli__subject__CAROLI-M-L-735: 12,961 rows
[11/354] distance PCA availability Mus_caroli__subject__CAROLI-M-L-738: 12,961 rows
[12/354] distance PCA availability Mus_caroli__subject__CAROLI-M-LR-734: 8,641 rows


,species,available_sample_rows,n_sample_rows
0,Mus_caroli,185775,60480
1,Mus_musculus,1715136,60480
2,Mus_spretus,194416,60480
3,Peromyscus_californicus,103680,60480
4,Peromyscus_gossypinus,86400,60480
5,Peromyscus_leucopus,60480,60480
6,Peromyscus_maniculatus,635040,60480
7,Peromyscus_polionotus,211680,60480


pre-wavelet covariance PCA: 100 components from 300 features
[1/354] covariance sample Mus_caroli__subject__CAROLI-F-2L-741: (2813, 300)
    covariance batch 1: (2813, 300), rows seen=2,813
[2/354] covariance sample Mus_caroli__subject__CAROLI-F-L-737: (4220, 300)
    covariance batch 2: (4220, 300), rows seen=7,033
[3/354] covariance sample Mus_caroli__subject__CAROLI-F-L-741: (4220, 300)
    covariance batch 3: (4220, 300), rows seen=11,253
[4/354] covariance sample Mus_caroli__subject__CAROLI-F-LR-741: (4220, 300)
    covariance batch 4: (4220, 300), rows seen=15,473
[5/354] covariance sample Mus_caroli__subject__CAROLI-F-N-740: (4220, 300)
    covariance batch 5: (4220, 300), rows seen=19,693
[6/354] covariance sample Mus_caroli__subject__CAROLI-F-N-741: (4220, 300)
    covariance batch 6: (4220, 300), rows seen=23,913
[7/354] covariance sample Mus_caroli__subject__CAROLI-F-R-737: (4220, 300)
    covariance batch 7: (4220, 300), rows seen=28,133
[8/354] covariance sample Mus_caroli

In [5]:
def save_shared_pca_loadings_per_species(
    pca,
    n_components,
    global_ids,
    output_path,
    feature_names=None,
):
    """
    Save shared PCA loadings in long format.

    Each row represents:
        species × principal component × original feature

    Because the PCA was fitted on pooled cross-species data, the loading values
    are identical for every species. Repeating them by species makes subsequent
    species-level joins and analyses straightforward.
    """
    components = np.asarray(
        pca.components_[:n_components],
        dtype=float,
    )

    n_features = components.shape[1]

    if feature_names is None:
        feature_names = [f'feature_{i}' for i in range(n_features)]
    else:
        feature_names = list(feature_names)

    if len(feature_names) != n_features:
        raise ValueError(
            f'Expected {n_features} feature names, '
            f'but received {len(feature_names)}.'
        )

    species_names = sorted({
        parse_global_id(global_id)[0]
        for global_id in global_ids
    })

    loading_rows = []

    for species in species_names:
        for pc_index in range(n_components):
            explained_variance = float(
                pca.explained_variance_[pc_index]
            )
            explained_variance_ratio = float(
                pca.explained_variance_ratio_[pc_index]
            )
            cumulative_variance_ratio = float(
                np.sum(pca.explained_variance_ratio_[:pc_index + 1])
            )

            for feature_index, feature_name in enumerate(feature_names):
                loading = float(components[pc_index, feature_index])

                loading_rows.append({
                    'species': species,
                    'component': pc_index + 1,
                    'component_label': f'PC{pc_index + 1}',
                    'feature_index': feature_index,
                    'feature_name': feature_name,
                    'loading': loading,
                    'absolute_loading': abs(loading),
                    'squared_loading': loading ** 2,
                    'explained_variance': explained_variance,
                    'explained_variance_ratio': explained_variance_ratio,
                    'cumulative_variance_ratio': cumulative_variance_ratio,
                })

    loadings_df = pd.DataFrame(loading_rows)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    loadings_df.to_csv(output_path, index=False)

    print(
        f'Saved shared PCA loadings for {len(species_names)} species: '
        f'{output_path}'
    )
    print(
        f'CSV shape: {loadings_df.shape}; '
        f'{n_components} PCs × {n_features} features × '
        f'{len(species_names)} species'
    )

    return loadings_df


shared_pca_loadings = save_shared_pca_loadings_per_species(
    pca=pre_wavelet_pca,
    n_components=pre_wavelet_n_components,
    global_ids=global_ids,
    output_path=RUN_ROOT / 'pre_wavelet_shared_pca_loadings_per_species.csv',
)

Saved shared PCA loadings for 8 species: /Users/meganbishop/slowmodeevo/outputs/single_species_distance_comparison/pca_multispecies_evensamp/pre_wavelet_shared_pca_loadings_per_species.csv
CSV shape: (67200, 11); 28 PCs × 300 features × 8 species


## 5. Wavelet Transform And PCA After Wavelets

In [6]:
def load_pre_wavelet_pca_trace(pipeline_species, individual_id, dataset_name=None):
    species, source_individual_id = parse_global_id(individual_id)
    bundle = base_loader(species, individual_id=source_individual_id, dataset_name=REPRESENTATION_NAME)
    transformed = pre_wavelet_pca.transform(bundle['X']).astype(np.float32, copy=False)
    bundle['X'] = transformed[:, :pre_wavelet_n_components]
    bundle['species_folder'] = species
    bundle['source_individual_id'] = source_individual_id
    bundle['individual_id'] = str(individual_id)
    bundle['dataset_name'] = dataset_name or DATASET_NAME_AFTER_PRE_PCA
    bundle['pre_wavelet_pca_checkpoint'] = str(RUN_ROOT / 'pre_wavelet_pca95_checkpoint.pkl')
    return bundle

projection_result = pup.prepare_pooled_individual_projections(
    pipeline_species='all_species',
    individual_ids=global_ids,
    load_individual_trace_fn=load_pre_wavelet_pca_trace,
    dataset_name=DATASET_NAME_AFTER_PRE_PCA,
    fs=FS_HZ,
    fmin=FMIN,
    fmax=FMAX,
    n_freqs=N_FREQS,
    output_dir=RUN_ROOT,
    wavelet_subsample_factor=WAVELET_SUBSAMPLE_FACTOR,
    pca_max_components=POST_WAVELET_PCA_MAX_COMPONENTS,
    n_shuffles=5,
    pca_variance_target=POST_WAVELET_PCA_VARIANCE_TARGET,
    wavelet_dtype=np.float32,
    normalize_wavelets_for_pca=NORMALIZE_WAVELETS_FOR_PCA,
    balance_pca_samples_by_species=BALANCE_POST_WAVELET_PCA_SAMPLES_BY_SPECIES,
    pca_sample_species_labels=[parse_global_id(global_id)[0] for global_id in global_ids],
    pca_rows_per_species=POST_WAVELET_PCA_ROWS_PER_SPECIES,
    reuse_existing_wavelet_samples=REUSE_EXISTING,
    reuse_existing_projections=REUSE_EXISTING,
    save_wavelet_samples=SAVE_WAVELET_SAMPLES,
    seed=SEED,
    verbose=True,
)

metadata_by_global_id = individual_manifest.set_index('global_id').to_dict('index')
for meta in projection_result['metadata']:
    extra = metadata_by_global_id[str(meta['individual_id'])]
    meta['species'] = extra['species']
    meta['source_individual_id'] = extra['individual_id']
projection_result['input_preprocessing'] = 'pre_wavelet_pca_then_wavelet_then_post_wavelet_pca'
projection_result['pre_wavelet_pca_checkpoint'] = str(RUN_ROOT / 'pre_wavelet_pca95_checkpoint.pkl')
projection_result['pre_wavelet_pca_n_components'] = pre_wavelet_n_components
projection_result['pre_wavelet_pca_variance_target'] = float(DISTANCE_PCA_VARIANCE_TARGET)
projection_result['pre_wavelet_pca_balance_samples_by_species'] = bool(BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES)
projection_result['post_wavelet_pca_balance_samples_by_species'] = bool(BALANCE_POST_WAVELET_PCA_SAMPLES_BY_SPECIES)
projection_result['pre_wavelet_pca_fit_method'] = DISTANCE_PCA_FIT_METHOD
projection_result['post_wavelet_pca_fit_method'] = POST_WAVELET_PCA_FIT_METHOD
projection_result['run_root'] = str(RUN_ROOT)
with open(RUN_ROOT / 'projection_result.pkl', 'wb') as handle:
    pickle.dump(projection_result, handle)

pd.DataFrame({
    'component': np.arange(1, len(distance_pca_result['cumulative_variance']) + 1),
    'cumulative_variance': np.asarray(distance_pca_result['cumulative_variance'], dtype=float),
}).to_csv(RUN_ROOT / 'pre_wavelet_pca_cumulative_variance.csv', index=False)

post_cumulative = projection_result.get('pca_cumulative_variance')
if post_cumulative is not None:
    pd.DataFrame({
        'component': np.arange(1, len(post_cumulative) + 1),
        'cumulative_variance': np.asarray(post_cumulative, dtype=float),
    }).to_csv(RUN_ROOT / 'post_wavelet_pca_cumulative_variance.csv', index=False)

projection_rows = []
for meta, proj_file in zip(projection_result['metadata'], projection_result['proj_files']):
    arr = np.load(proj_file, mmap_mode='r')
    projection_rows.append({
        'global_id': str(meta['individual_id']),
        'species': meta.get('species', ''),
        'source_individual_id': meta.get('source_individual_id', ''),
        'projection_file': str(Path(proj_file).resolve()),
        'n_frames': int(arr.shape[0]),
        'n_projection_dims': int(arr.shape[1]),
    })
projection_file_manifest = pd.DataFrame(projection_rows)
projection_file_manifest.to_csv(RUN_ROOT / 'projection_files_manifest.csv', index=False)

summary = pd.DataFrame([{
    'run_name': RUN_NAME,
    'run_root': str(RUN_ROOT),
    'n_species': len(species_names),
    'n_individuals': len(global_ids),
    'pre_wavelet_pca_components': pre_wavelet_n_components,
    'pre_wavelet_pca_variance': float(distance_pca_result['cumulative_variance'][pre_wavelet_n_components - 1]),
    'post_wavelet_pca_components': int(projection_result['n_kept']),
    'post_wavelet_pca_selection_mode': projection_result.get('pca_selection_mode'),
    'post_wavelet_pca_variance_target': projection_result.get('pca_variance_target'),
    'post_wavelet_pca_target_reached': projection_result.get('pca_variance_target_reached'),
    'wavelet_subsample_factor': WAVELET_SUBSAMPLE_FACTOR,
    'normalize_wavelets_for_pca': NORMALIZE_WAVELETS_FOR_PCA,
    'pre_wavelet_pca_balance_samples_by_species': bool(BALANCE_DISTANCE_PCA_SAMPLES_BY_SPECIES),
    'post_wavelet_pca_balance_samples_by_species': bool(BALANCE_POST_WAVELET_PCA_SAMPLES_BY_SPECIES),
    'pre_wavelet_pca_fit_method': DISTANCE_PCA_FIT_METHOD,
    'post_wavelet_pca_fit_method': POST_WAVELET_PCA_FIT_METHOD,
    'projection_result': str(RUN_ROOT / 'projection_result.pkl'),
    'projection_files_manifest': str(RUN_ROOT / 'projection_files_manifest.csv'),
}])
summary.to_csv(RUN_ROOT / 'projection_generation_summary.csv', index=False)
display(summary)
display(projection_file_manifest.head())


Pass 1/3: checkpoint each individual's sampled wavelet rows (every 50 row)
  [1/354] Mus_caroli__subject__CAROLI-F-2L-741: X=(432016, 28)
  [2/354] Mus_caroli__subject__CAROLI-F-L-737: X=(648024, 28)
  [3/354] Mus_caroli__subject__CAROLI-F-L-741: X=(648024, 28)
  [4/354] Mus_caroli__subject__CAROLI-F-LR-741: X=(648024, 28)
  [5/354] Mus_caroli__subject__CAROLI-F-N-740: X=(648024, 28)
  [6/354] Mus_caroli__subject__CAROLI-F-N-741: X=(648024, 28)
  [7/354] Mus_caroli__subject__CAROLI-F-R-737: X=(648024, 28)
  [8/354] Mus_caroli__subject__CAROLI-F-R-741: X=(648024, 28)
  [9/354] Mus_caroli__subject__CAROLI-M-L-734: X=(648024, 28)
  [10/354] Mus_caroli__subject__CAROLI-M-L-735: X=(648024, 28)
  [11/354] Mus_caroli__subject__CAROLI-M-L-738: X=(648024, 28)
  [12/354] Mus_caroli__subject__CAROLI-M-LR-734: X=(432016, 28)
  [13/354] Mus_caroli__subject__CAROLI-M-N-739: X=(648024, 28)
  [14/354] Mus_caroli__subject__CAROLI-M-R-734: X=(648024, 28)
  [15/354] Mus_caroli__subject__CAROLI-M-R-738: X

,run_name,run_root,n_species,n_individuals,pre_wavelet_pca_components,pre_wavelet_pca_variance,post_wavelet_pca_components,post_wavelet_pca_selection_mode,post_wavelet_pca_variance_target,post_wavelet_pca_target_reached,wavelet_subsample_factor,normalize_wavelets_for_pca,pre_wavelet_pca_balance_samples_by_species,post_wavelet_pca_balance_samples_by_species,pre_wavelet_pca_fit_method,post_wavelet_pca_fit_method,projection_result,projection_files_manifest
0,pca_multispecies_evensamp,/Users/meganbishop/slowmodeevo/outputs/single_...,8,354,28,0.990521,100,variance_target,0.99,False,50,True,True,True,streaming_covariance,streaming_covariance,/Users/meganbishop/slowmodeevo/outputs/single_...,/Users/meganbishop/slowmodeevo/outputs/single_...


,global_id,species,source_individual_id,projection_file,n_frames,n_projection_dims
0,Mus_caroli__subject__CAROLI-F-2L-741,Mus_caroli,CAROLI-F-2L-741,/Users/meganbishop/slowmodeevo/outputs/single_...,432016,100
1,Mus_caroli__subject__CAROLI-F-L-737,Mus_caroli,CAROLI-F-L-737,/Users/meganbishop/slowmodeevo/outputs/single_...,648024,100
2,Mus_caroli__subject__CAROLI-F-L-741,Mus_caroli,CAROLI-F-L-741,/Users/meganbishop/slowmodeevo/outputs/single_...,648024,100
3,Mus_caroli__subject__CAROLI-F-LR-741,Mus_caroli,CAROLI-F-LR-741,/Users/meganbishop/slowmodeevo/outputs/single_...,648024,100
4,Mus_caroli__subject__CAROLI-F-N-740,Mus_caroli,CAROLI-F-N-740,/Users/meganbishop/slowmodeevo/outputs/single_...,648024,100


## 6. Optional Slow-Mode Estimation From Saved Projections

In [7]:
if RUN_SLOW_MODES:
    proj_files = [str(Path(path)) for path in projection_result['proj_files']]
    state_result = pup.assign_pooled_states_from_projections(
        proj_files=proj_files,
        d_embed=D_EMBED,
        N=N_CLUSTERS,
        output_dir=RUN_ROOT,
        kmeans_subsample_factor=KMEANS_SUBSAMPLE_FACTOR,
        seed=SEED,
        kmeans_n_init=KMEANS_N_INIT,
        save_states=True,
        verbose=True,
    )
    slow_result = pup.compute_pooled_slow_modes(
        state_result['states_list'],
        fs=FS_HZ,
        tau_seconds=TAU_SECONDS,
        N=N_CLUSTERS,
        M=N_BASINS,
        use_recursive_arms=USE_RECURSIVE_ARMS,
    )
    final_dir = RUN_ROOT / 'final_outputs'
    saved = pup.save_pooled_outputs(
        output_dir=final_dir,
        species='all_species',
        individual_ids=global_ids,
        state_files=state_result['state_files'],
        states_list=state_result['states_list'],
        T=slow_result['T'],
        pi=slow_result['pi'],
        evals=slow_result['evals'],
        evecs=slow_result['evecs'],
        chi=slow_result['gpcca']['chi'],
        parameters={
            'run_name': RUN_NAME,
            'dataset': DATASET_NAME_AFTER_PRE_PCA,
            'fs': FS_HZ,
            'fmin': FMIN,
            'fmax': FMAX,
            'n_freqs': N_FREQS,
            'pre_wavelet_pca_components': pre_wavelet_n_components,
            'post_wavelet_pca_components': int(projection_result['n_kept']),
            'd_embed': D_EMBED,
            'N': N_CLUSTERS,
            'kmeans_subsample_factor': KMEANS_SUBSAMPLE_FACTOR,
            'tau_seconds': TAU_SECONDS,
            'lag_frames': slow_result['lag'],
            'M': N_BASINS,
            'use_recursive_arms': USE_RECURSIVE_ARMS,
        },
        metadata=projection_result['metadata'],
        arm_tree=slow_result.get('arm_tree'),
    )
    with open(RUN_ROOT / 'slow_mode_result.pkl', 'wb') as handle:
        pickle.dump({
            'state_files': state_result['state_files'],
            'd_embed': D_EMBED,
            'N': N_CLUSTERS,
            'kmeans_subsample_factor': KMEANS_SUBSAMPLE_FACTOR,
            'lag': slow_result['lag'],
            'tau_seconds': TAU_SECONDS,
            'M': N_BASINS,
            'final_outputs': str(final_dir),
            'saved': saved,
        }, handle)
    print(f'Saved slow-mode outputs under {final_dir}')
else:
    print('RUN_SLOW_MODES is False; projections are ready for clustering notebooks.')

Fitting shared k-means: d_embed=10, N=1000, sample every 20 embedded rows
  k-means sample matrix = (7981351, 1000)
Assigning every embedded frame to shared clusters
  [1/354] Mus_caroli__subject__CAROLI-F-2L-741: assigning states
  [2/354] Mus_caroli__subject__CAROLI-F-L-737: assigning states
  [3/354] Mus_caroli__subject__CAROLI-F-L-741: assigning states
  [4/354] Mus_caroli__subject__CAROLI-F-LR-741: assigning states
  [5/354] Mus_caroli__subject__CAROLI-F-N-740: assigning states
  [6/354] Mus_caroli__subject__CAROLI-F-N-741: assigning states
  [7/354] Mus_caroli__subject__CAROLI-F-R-737: assigning states
  [8/354] Mus_caroli__subject__CAROLI-F-R-741: assigning states
  [9/354] Mus_caroli__subject__CAROLI-M-L-734: assigning states
  [10/354] Mus_caroli__subject__CAROLI-M-L-735: assigning states
  [11/354] Mus_caroli__subject__CAROLI-M-L-738: assigning states
  [12/354] Mus_caroli__subject__CAROLI-M-LR-734: assigning states
  [13/354] Mus_caroli__subject__CAROLI-M-N-739: assigning st